# NewYork City waste history dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

## 1. Download dataset

In [2]:
DATA_URL = "https://data.cityofnewyork.us/api/views/ebb7-mvp5/rows.csv?accessType=DOWNLOAD"

df = pd.read_csv(DATA_URL)
df.head()

,MONTH,BOROUGH,COMMUNITYDISTRICT,REFUSETONSCOLLECTED,PAPERTONSCOLLECTED,MGPTONSCOLLECTED,RESORGANICSTONS,SCHOOLORGANICTONS,LEAVESORGANICTONS,XMASTREETONS,OTHERORGANICSTONS,BOROUGH_ID
0,2026 / 03,Bronx,1,3511.8,295.6,173.3,22.1,86.0,NaN,NaN,NaN,2
1,2026 / 03,Bronx,2,8177.7,229.1,253.3,11.0,100.1,NaN,NaN,NaN,2
2,2026 / 03,Bronx,3,2447.5,177.3,170.8,16.9,NaN,NaN,NaN,NaN,2
3,2026 / 03,Bronx,4,4369.3,310.2,347.7,15.0,70.4,NaN,NaN,NaN,2
4,2026 / 03,Bronx,5,3712.2,289.7,359.6,13.5,74.9,NaN,NaN,NaN,2


## 2. Primary analysis

In [3]:
print(df.shape)
df.columns.tolist()

(24942, 12)


['MONTH',
 'BOROUGH',
 'COMMUNITYDISTRICT',
 'REFUSETONSCOLLECTED',
 'PAPERTONSCOLLECTED',
 'MGPTONSCOLLECTED',
 'RESORGANICSTONS',
 'SCHOOLORGANICTONS',
 'LEAVESORGANICTONS',
 'XMASTREETONS',
 'OTHERORGANICSTONS',
 'BOROUGH_ID']

In [4]:
df.info()

missing = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)
missing[missing["missing_count"] > 0].head(20)

<class 'pandas.DataFrame'>
RangeIndex: 24942 entries, 0 to 24941
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   MONTH                24942 non-null  str    
 1   BOROUGH              24942 non-null  str    
 2   COMMUNITYDISTRICT    24942 non-null  int64  
 3   REFUSETONSCOLLECTED  24942 non-null  float64
 4   PAPERTONSCOLLECTED   22508 non-null  float64
 5   MGPTONSCOLLECTED     22865 non-null  float64
 6   RESORGANICSTONS      3390 non-null   float64
 7   SCHOOLORGANICTONS    2497 non-null   float64
 8   LEAVESORGANICTONS    881 non-null    float64
 9   XMASTREETONS         1685 non-null   float64
 10  OTHERORGANICSTONS    1254 non-null   float64
 11  BOROUGH_ID           24942 non-null  int64  
dtypes: float64(8), int64(2), str(2)
memory usage: 2.7 MB


,missing_count
LEAVESORGANICTONS,24061
OTHERORGANICSTONS,23688
XMASTREETONS,23257
SCHOOLORGANICTONS,22445
RESORGANICSTONS,21552
PAPERTONSCOLLECTED,2434
MGPTONSCOLLECTED,2077


In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
MONTH,24942,431,2026 / 03,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BOROUGH,24942,5,Brooklyn,7551,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COMMUNITYDISTRICT,24942.0,NaN,NaN,NaN,7.417689,4.477881,1.0,3.0,7.0,11.0,18.0
REFUSETONSCOLLECTED,24942.0,NaN,NaN,NaN,4239.402907,1553.373854,8.4,3126.7,4061.85,5194.6,10413.9
PAPERTONSCOLLECTED,22508.0,NaN,NaN,NaN,447.931273,270.486,0.0,258.275,399.0,583.9,2129.7
MGPTONSCOLLECTED,22865.0,NaN,NaN,NaN,353.721251,181.236664,0.0,222.4,334.1,457.9,1254.6
RESORGANICSTONS,3390.0,NaN,NaN,NaN,89.420531,90.3651,0.4,27.8,61.7,117.575,785.5
SCHOOLORGANICTONS,2497.0,NaN,NaN,NaN,54.169403,38.157595,0.1,28.3,43.2,70.1,265.7
LEAVESORGANICTONS,881.0,NaN,NaN,NaN,166.229171,268.619456,0.6,12.9,58.6,197.3,1857.3
XMASTREETONS,1685.0,NaN,NaN,NaN,26.272107,25.108075,0.4,9.6,18.0,35.1,186.0


## 3. Standardize columns naming

In [6]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_", regex=False)
)

df.columns.tolist()

['month',
 'borough',
 'communitydistrict',
 'refusetonscollected',
 'papertonscollected',
 'mgptonscollected',
 'resorganicstons',
 'schoolorganictons',
 'leavesorganictons',
 'xmastreetons',
 'otherorganicstons',
 'borough_id']

In [7]:
candidate_cols = [col for col in df.columns if any(
    key in col for key in ["month", "year", "borough", "district", "tons", "organics", "refuse", "paper"]
)]
candidate_cols

['month',
 'borough',
 'communitydistrict',
 'refusetonscollected',
 'papertonscollected',
 'mgptonscollected',
 'resorganicstons',
 'schoolorganictons',
 'leavesorganictons',
 'xmastreetons',
 'otherorganicstons',
 'borough_id']

In [8]:
df["month"] = pd.to_datetime(df["month"], errors="coerce")
df["year"] = df["month"].dt.year
df["month_num"] = df["month"].dt.month

df[["month", "year", "month_num"]].head()

/tmp/ipykernel_10784/1695219150.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["month"] = pd.to_datetime(df["month"], errors="coerce")


,month,year,month_num
0,2026-03-01,2026,3
1,2026-03-01,2026,3
2,2026-03-01,2026,3
3,2026-03-01,2026,3
4,2026-03-01,2026,3


In [ ]:
print("Doublons complets :", df.duplicated().sum())

for col in ["borough", "communitydistrict"]:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} valeurs uniques")
        print(df[col].dropna().unique()[:10])
        print()

In [ ]:
target_cols = [col for col in df.columns if "tonscollected" in col or "ton" in col]
target_cols[:10]

In [ ]:
if "refusetonscollected" in df.columns:
    plt.figure(figsize=(14, 5))
    (
        df.groupby("month")["refusetonscollected"]
        .sum()
        .sort_index()
        .plot()
    )
    plt.title("Refuse tons collected over time")
    plt.xlabel("Month")
    plt.ylabel("Tons")
    plt.show()